# Safe automation by demographic group in an LLM referral system

The model answers a clinical question and reports a confidence score; answers
above a threshold are accepted automatically and the rest are referred to a
clinician. For an error budget α, the **safe automation rate (SAR)** of a group
is the largest share of its cases that can be accepted with error ≤ α.

This notebook runs the open-weight models on a GPU (Colab L4 or A100) and then
performs the full analysis, merging in the commercial-model results produced by
`notebooks/api_models.ipynb`.

| | |
|---|---|
| Pairs | 1,000 matched pairs per dataset and attribute (MedQA and MedMCQA; race and sex), built by rule so that the two versions differ only in the demographic descriptor |
| Models | MedGemma-4B, Med42-8B, Gemma-4-E4B, Llama-3.1-8B-Instruct; token-probability (TLP) then elicited (CE) confidence |
| Commercial models | GPT-6-Sol, GPT-5.6-Terra, GPT-4.1-mini, Claude-Sonnet-5 (MedQA race) |
| Prompts | P1 bare · P2 bias-aware · P3 equity-aware |

Every response is cached as soon as it is produced, so an interrupted run
resumes where it stopped. The code is in `care/` and is covered by `tests/`.

## 0. Colab setup

**Runtime → Change runtime type → L4 or A100 GPU.** A free T4 has 15 GB, which
cannot hold the 8B models in 16-bit, and it lacks native bf16, which Gemma needs.
Quantising to fit would change the confidence scores being measured, so it is
not an option here.

Before running:

1. Put the MedQA and MedMCQA source files in `My Drive/CARE/source/` (see the README); the
   matched pairs are built from them on the first run. 
2. In the key icon (Secrets) on the left, add `HF_TOKEN` and enable notebook
   access. API keys are not needed here: the API models run in `notebooks/api_models.ipynb`.
   When that is done, upload its `api_results.zip` to `My Drive/CARE/` too.
3. Accept the model terms on Hugging Face for Llama-3.1 and MedGemma 4B (`google/medgemma-4b-it`).

All inference caches and outputs are written to Drive, so a disconnected
session loses nothing: re-run from the top and it resumes where it stopped.
Keep the tab open on the free tier; Colab Pro+ can run in the background.

In [ ]:
!pip -q install "transformers>=5.8" accelerate openai anthropic
import os, subprocess
from google.colab import drive, userdata
drive.mount("/content/drive")

REPO_URL = "https://github.com/haohuilu/CARE.git"
CODE = "/content/CARE"
if not os.path.isdir(CODE):
    subprocess.run(["git", "clone", "-q", REPO_URL, CODE], check=True)

# Everything that must survive a disconnect lives on Drive; set before `care` is imported.
DRIVE = "/content/drive/MyDrive/CARE"
os.environ["CARE_SOURCE_DIR"] = f"{DRIVE}/source"    # MedQA/ and medmcqa/ (see README)
os.environ["CARE_PAIRS_DIR"] = f"{DRIVE}/pairs"
os.environ["CARE_DATA_ROOT"] = f"{DRIVE}/results"    # cached model responses
os.environ["CARE_OUTPUT_ROOT"] = f"{DRIVE}/outputs"  # tables and figures
# Results of notebooks/api_models.ipynb, if uploaded as a zip (separate model folders).
if os.path.exists(f"{DRIVE}/api_results.zip"):
    import zipfile
    zipfile.ZipFile(f"{DRIVE}/api_results.zip").extractall(f"{DRIVE}/results")
    print("unpacked api_results.zip into results/")
try:
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
    print("  OK       HF_TOKEN")
except Exception as e:
    print("  MISSING  HF_TOKEN -- key icon on the left > Add new secret, and enable notebook access")
os.chdir(CODE)

## 1. Setup

In [ ]:
import os, sys, json
from pathlib import Path
CARE = Path(CODE)
sys.path.insert(0, str(CARE)); os.chdir(CARE)

import pandas as pd
from care import config as cfg, infer
from care import analysis

RESULTS = Path(cfg.DATA_ROOT)
DEVICE, DTYPE = infer.pick_device()
print("device:", DEVICE, DTYPE, "| results ->", RESULTS)
infer.gpu_report(["medgemma", "Qwen3.5-9B", "Med42-8B"])

## 2. Pairs

In [ ]:
N_OPTIONS = {"pairs_medqa_race": 5, "pairs_medqa_gender": 5,
             "pairs_medmcqa_race": 4, "pairs_medmcqa_gender": 4}
PAIR_FILES = {d: Path(cfg.PAIRS_DIR) / f"{d}.jsonl" for d in N_OPTIONS}
if not all(p.exists() for p in PAIR_FILES.values()):         # build once from the source corpora
    subprocess.run([sys.executable, "-m", "care.pairgen"], check=True)

PAIRS = {d: [json.loads(l) for l in open(p, encoding="utf-8") if l.strip()]
         for d, p in PAIR_FILES.items()}
for d, r in PAIRS.items():
    print(f"{d:22s} {len(r)//2:5d} pairs  groups={sorted({x['demographic'] for x in r})}")
print(f"\n{sum(map(len, PAIRS.values())):,} items per model per condition")

Look at a few pairs before spending compute. Differing words are in [brackets] and the gold option is marked. Call `show_pair("pairs_medqa_race")` repeatedly for random pairs, or pass `i=` for a fixed one.

In [ ]:
import difflib, random

def show_pair(dataset, i=None, width=100):
    """Print both arms of one minimal pair, with the differing words in [brackets]."""
    recs = PAIRS[dataset]
    ids = sorted({r["pair_id"] for r in recs})
    pid = ids[i] if i is not None else random.choice(ids)
    a, b = [r for r in recs if r["pair_id"] == pid]
    ta, tb = a["question"].split(), b["question"].split()
    ma, mb = [], []
    for op, i1, i2, j1, j2 in difflib.SequenceMatcher(None, ta, tb).get_opcodes():
        ma += ta[i1:i2] if op == "equal" else [f"[{w}]" for w in ta[i1:i2]]
        mb += tb[j1:j2] if op == "equal" else [f"[{w}]" for w in tb[j1:j2]]
    print(f"=== {dataset} | pair {pid}" + (f" | {a['subject']}" if a.get("subject") else "") + " ===")
    for rec, words in ((a, ma), (b, mb)):
        print(f"\n[{rec['demographic'].upper()}]")
        line = ""
        for w in words:                               # simple word wrap
            if len(line) + len(w) > width:
                print(" ", line); line = ""
            line += w + " "
        print(" ", line)
    print("\nOptions:")
    for k, v in a["options"].items():
        print(f"  {k}: {v}" + ("   <-- gold" if k == a["answer_idx"] else ""))
    print()

# one example from each dataset
for ds in PAIRS:
    show_pair(ds, i=0)

## 3. Prompts

Verbatim from the manuscript appendix (there C1, C3, C5), kept in
`care/prompts.py` so the GPU and API notebooks use identical text. The
clinician-role and epidemiological-context prompts are not run.

In [ ]:
from care.prompts import SYSTEM, CONDITIONS    # one source for GPU and API runs
for c in CONDITIONS:
    print(f"--- {c} ---\n{SYSTEM[c]}\n")
print(infer.user_prompt(PAIRS["pairs_medqa_race"][0])[-230:])

## 4. Local models (main text)

Each model runs **TLP first, then CE** on the same loaded weights. CE needs generation, so it takes roughly 2-3x as long as TLP.

Gated models (Llama, MedGemma) need `huggingface-cli login` and accepted terms.
The smoke test checks four things per model before the full run: accuracy is
plausible, logits are finite, **letter mass** is high (the model puts its
next-token probability on an option letter — if not, TLP is not a valid
confidence for that model), and batched results match unbatched ones.

In [ ]:
LOCAL = {                                   # HF id -> results directory
    "google/medgemma-4b-it":      "google_medgemma-4b-it",       # medical, May 2025; answers directly
    # "google/medgemma-1.5-4b-it": reasons before answering; forced to answer directly it is near
    #   chance with <5% probability on option letters (pilot + diagnostic), so it is not used
    "google/gemma-4-E4B-it":      "google_gemma-4-E4B-it",       # general, Mar 2026; standard attention, fast
    "m42-health/Llama3-Med42-8B": "m42-health_Llama3-Med42-8B",  # medical, continuity with earlier runs
    "meta-llama/Llama-3.1-8B-Instruct": "meta-llama_Llama-3.1-8B-Instruct",  # general, continuity
    # "google/medgemma-27b-text-it": "google_medgemma-27b-text-it",  # A100 80GB only
}
BATCH = {}                                  # per-model override; default below
DEFAULT_BATCH = 8 if DEVICE != "cuda" else 16
META = {}                                   # filled in for the manifest
APPENDIX, APPX_METHODS, APPX_DATA = {}, {}, {}   # set in the appendix section (optional)
import gc, torch
def free(model=None):
    """Release a model's GPU memory before loading the next one."""
    del model
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

from transformers import AutoConfig
for name in LOCAL:
    try:
        AutoConfig.from_pretrained(name); print("  OK      ", name)
    except Exception as e:
        print("  NO ACCESS", name, "->", str(e)[:80])

In [ ]:
import time
def smoke(name, n=16):
    tok, model = infer.load_local(name, DEVICE, DTYPE)
    ids = infer.letter_token_ids(tok)
    recs = PAIRS["pairs_medqa_race"][:n]
    texts = [infer.chat_text(tok, SYSTEM["P1"], infer.user_prompt(r)) for r in recs]
    t0 = time.time()
    batched = infer.tlp_batch(model, tok, ids, texts, infer.LETTERS[:5])
    dt = (time.time() - t0) / n
    single = [infer.tlp_batch(model, tok, ids, [t], infer.LETTERS[:5])[0] for t in texts[:4]]
    diff = max(abs(a["probs"][k] - b["probs"][k]) for a, b in zip(single, batched) for k in a["probs"])
    acc = sum(r["pred"] == x["answer_idx"] for r, x in zip(batched, recs)) / n
    mass = sorted(r["letter_mass"] for r in batched)[n // 2]
    total = sum(map(len, PAIRS.values())) * len(CONDITIONS)
    print(f"{name}\n  accuracy {acc:.2f} | median letter mass {mass:.2f} | "
          f"batch-vs-single diff {diff:.1e} | {dt:.3f}s/item -> ~{total*dt/3600:.1f} h")
    if acc < 0.35:
        print("  WARNING: accuracy near chance (0.20) -- check the load log for newly "
              "initialised weights or a wrong chat template before the full run")
    if mass < 0.5:
        print("  WARNING: low letter mass -- TLP is not a faithful confidence for this model")
        print("  what it wants to write first:", infer.top_next_tokens(model, tok, texts[0]))
    if diff > 0.02:
        # An item's confidence must not depend on which other items share its
        # batch; single-item runs are deterministic, so fall back to them.
        BATCH[name] = 1
        print(f"  batch-vs-single diff {diff:.3f} > 0.02 -> this model runs with batch size 1")
    ce_texts = [infer.chat_text(tok, SYSTEM["P1"], infer.user_prompt(r, "ce")) for r in recs]
    t0 = time.time()
    ce = infer.ce_batch(model, tok, ce_texts, infer.LETTERS[:5])
    dt_ce = (time.time() - t0) / n
    parsed = sum(r["answer"] is not None for r in ce) / n
    print(f"  CE: parsed {parsed:.0%} | {dt_ce:.3f}s/item -> ~{total*dt_ce/3600:.1f} h | "
          f"e.g. {ce[0]['raw'][:40]!r}")
    if parsed < 0.9:
        print("  WARNING: CE replies often unparsable -- check the examples above")
    free(model)

for name in LOCAL:
    try:
        smoke(name)
    except Exception as e:
        print(f"{name}\n  FAILED: {type(e).__name__}: {str(e)[:300]}")
        free()

### Pilot: 20 pairs per dataset, all prompts

The smoke test uses 16 MedQA items under P1 only. The pilot runs every dataset
(including 4-option MedMCQA) under all three prompts, TLP and CE, and checks the
parse rate, accuracy and letter mass per slice before committing hours of GPU.
Pilot items are part of the full run, so nothing is computed twice.

In [ ]:
from care.data import load_records

def pilot_pairs(n_pairs=20):
    out = {}
    for d, recs in PAIRS.items():
        keep = set(list(dict.fromkeys(r["pair_id"] for r in recs))[:n_pairs])
        out[d] = [r for r in recs if r["pair_id"] in keep]
    return out

PILOT = pilot_pairs(20)
for name, mdir in LOCAL.items():
    print(f"\n=== {name} ===")
    try:
        tok, model = infer.load_local(name, DEVICE, DTYPE)
        bs = BATCH.get(name, DEFAULT_BATCH)
        infer.run_local_tlp(model, tok, name, mdir, PILOT, N_OPTIONS, SYSTEM,
                            CONDITIONS, RESULTS, batch_size=bs)
        infer.run_local_ce(model, tok, name, mdir, PILOT, N_OPTIONS, SYSTEM,
                           CONDITIONS, RESULTS, batch_size=bs)
    except Exception as e:
        print(f"  FAILED: {type(e).__name__}: {str(e)[:300]}")
    finally:
        model = None
        free()

rows, flags = [], []
for name, mdir in LOCAL.items():
    for ds in PILOT:
        want = {(str(r["question_id"]), str(r["variant"]).lower()) for r in PILOT[ds]}
        for meth in ("tlp", "ce"):
            try:
                df = load_records(ds, mdir, meth)
            except FileNotFoundError:
                continue
            df = df[[(q, v) in want for q, v in zip(df.question_id, df.variant)]]
            for c, s in df.groupby("prompt"):
                r = {"model": cfg.short(mdir), "dataset": ds.replace("pairs_", ""),
                     "method": meth, "prompt": c, "n": len(s),
                     "parse": s.parse_ok.mean(), "acc": s.correct.mean(),
                     "mean_conf": s.conf_raw.mean(), "distinct": s.conf_raw.nunique(),
                     "letter_mass": s.letter_mass.median()}
                rows.append(r)
                chance = 1 / N_OPTIONS[ds]
                if r["parse"] < 0.9:
                    flags.append(f"{r['model']} {ds} {meth} {c}: parse rate {r['parse']:.2f}")
                if r["acc"] < chance + 0.1:
                    flags.append(f"{r['model']} {ds} {meth} {c}: accuracy {r['acc']:.2f} near chance {chance:.2f}")
                if meth == "tlp" and r["letter_mass"] < 0.5:
                    flags.append(f"{r['model']} {ds} tlp {c}: letter mass {r['letter_mass']:.2f}")
pilot = pd.DataFrame(rows)
with pd.option_context("display.max_rows", 500):
    print(pilot.to_string(index=False, float_format=lambda v: f"{v:.2f}"))
print("\nFLAGS:" if flags else "\nno flags -- go ahead with the full run")
for f in flags:
    print("  ", f)

# CE replies that could not be parsed (current instruction only), 3 per model
for name, mdir in LOCAL.items():
    bad, n = [], 0
    for ds in PILOT:
        for c in CONDITIONS:
            p = infer.out_path(RESULTS, ds, mdir, c)
            latest = {}                          # files are append-only: keep the newest reply
            for l in (open(p) if p.exists() else []):
                r = json.loads(l)
                if r.get("ce_version") == infer.CE_VERSION and "ce_think_prefix" in r:   # current code only
                    latest[(r["question_id"], r["variant"])] = r
            n += len(latest)
            bad += [r["ce_raw_text"] for r in latest.values()
                    if infer.parse_ce(r["ce_raw_text"], infer.LETTERS[:N_OPTIONS[ds]])[0] is None]
    print(f"\n{cfg.short(mdir)}: {len(bad)}/{n} CE replies unparsed")
    for b in bad[:3]:
        print("   ", repr(b[:120]))

### Full run

Go on only if the pilot shows no flags, or the flagged model is removed from `LOCAL`.

In [ ]:
for name, mdir in LOCAL.items():
    print(f"\n=== {name} ===")
    try:
        tok, model = infer.load_local(name, DEVICE, DTYPE)
        META[name] = {"backend": "local", "dir": mdir, "dtype": str(DTYPE),
                      "methods": ["tlp", "ce"],
                      "revision": getattr(model.config, "_commit_hash", None)}
        bs = BATCH.get(name, DEFAULT_BATCH)
        infer.run_local_tlp(model, tok, name, mdir, PAIRS, N_OPTIONS, SYSTEM,
                            CONDITIONS, RESULTS, batch_size=bs)
        infer.run_local_ce(model, tok, name, mdir, PAIRS, N_OPTIONS, SYSTEM,
                           CONDITIONS, RESULTS, batch_size=bs)
    except Exception as e:           # one failing model must not stop the others
        print(f"  FAILED: {type(e).__name__}: {str(e)[:300]} -- re-run this cell to resume")
    finally:
        model = None
        free()


## 5. Appendix: API models (run in `notebooks/api_models.ipynb`)

The API models run on a laptop in `notebooks/api_models.ipynb` (no GPU needed):
gpt-5.6-terra, gpt-4.1-mini (TLP + CE) and claude-sonnet-5 (CE), on the same
four pair files as the local models. Their results
reach `results/` either directly through Google Drive for desktop, or as
`api_results.zip`, which the setup cell unpacks. This cell only registers them
for the analysis; the methods are read from what is on disk.

In [ ]:
from care.data import load_records
APPENDIX = {"gpt-6-sol": "openai", "gpt-5.6-terra": "openai",
            "gpt-4.1-mini-2025-04-14": "openai", "claude-sonnet-5": "anthropic"}
def methods_on_disk(model):
    found = []
    for meth in ("tlp", "ce"):
        for ds in N_OPTIONS:
            try:
                load_records(ds, model, meth); found.append(meth); break
            except FileNotFoundError:
                pass
    return tuple(found)

def datasets_on_disk():
    return [ds for ds in N_OPTIONS
            if any((RESULTS / ds / m).exists() for m in APPENDIX)]

APPX_DATA = {ds: PAIRS[ds] for ds in datasets_on_disk()}
print("API datasets on disk:", list(APPX_DATA) or "none yet")

APPX_METHODS = {m: methods_on_disk(m) for m in APPENDIX}
for m, meths in APPX_METHODS.items():
    print(f"{m:26s} {meths or 'no results yet -- run CARE_api.ipynb'}")
APPENDIX = {m: b for m, b in APPENDIX.items() if APPX_METHODS[m]}   # analyse what exists

## 6. Completeness, validation, manifest

An unparsable reply has no usable confidence, so the system refers it: it stays
in every denominator as a referral. A parse-rate difference of more than 5
points between groups is reported as a warning, because part of any SAR gap is
then parse failure. Incomplete pairs are left out so the design stays matched,
and records from another pair sample (an old run left on Drive) are flagged —
delete those files before analysing.

In [ ]:
from care.data import load_records, validate
rows = []
runs = [(d, m, meth) for d in N_OPTIONS for m in LOCAL.values() for meth in ("tlp", "ce")] + \
       [(d, m, meth) for m in APPENDIX for meth in APPX_METHODS.get(m, ("ce",)) for d in APPX_DATA]
for ds, mdir, meth in runs:
    try:
        rep = validate(load_records(ds, mdir, meth), strict=False)
        n = int(rep.per_group["n"].sum())
        rows.append({"dataset": ds, "model": mdir, "method": meth, "n": n,
                     "expected": len(PAIRS[ds]) * len(CONDITIONS),
                     "errors": "; ".join(rep.errors) or "-"})
    except FileNotFoundError:
        rows.append({"dataset": ds, "model": mdir, "method": meth, "n": 0,
                     "expected": len(PAIRS[ds]) * len(CONDITIONS), "errors": "not run"})
chk = pd.DataFrame(rows); chk["complete"] = chk.n >= chk.expected
print(chk.to_string(index=False))

# Records from an older pair sample (e.g. a previous run left on Drive) would be
# mixed into the analysis; they must be deleted before analysing.
stale = 0
for ds, mdir, meth in runs:
    want = {(str(r["question_id"]), str(r["variant"]).lower()) for r in PAIRS[ds]}
    try:
        df = load_records(ds, mdir, meth)
    except FileNotFoundError:
        continue
    bad = sum((q, str(v)) not in want for q, v in zip(df.question_id, df.variant))
    if bad:
        stale += 1
        print(f"  STALE  {ds} / {mdir} / {meth}: {bad} records not in the current pairs")
print("no stale records" if not stale else
      f"WARNING: {stale} runs contain records from another pair sample -- delete those "
      f"files under {RESULTS} and re-run them")

print(infer.write_manifest(Path(cfg.RESULTS_DIR) / "run_manifest.json",
                           {d: str(p) for d, p in PAIR_FILES.items()}, SYSTEM, META))

## 7. Results

Read in this order: **SAR per group at α = 0.10** and whether the demographic
group is at zero; then the **P2/P3 vs P1 contrast**. For the negative claim,
`rules_out_meaningful_improvement` must be true — otherwise the honest statement
is only "no evidence that prompting helps".

SAR picks each threshold on the cases it is scored on. The **out-of-sample
check** sets the threshold on 4/5 of the pairs and applies it to the other 1/5,
as a hospital would after an audit: `cf_cov` should be close to SAR and
`cf_err` close to α. Report both; a large drop is the honest deployment figure.

In [ ]:
res = analysis.run(datasets=list(N_OPTIONS), models=list(LOCAL.values()), methods=("tlp", "ce"),
              tag="main")
sar, con = res["sar"], res["contrast"]
view = sar[sar.alpha == 0.10]
cols = ["dataset", "model", "acc_ref", "acc_dem", "ece_ref", "ece_dem",
        "sar_ref", "sar_dem", "sar_gap", "sar_gap_lo", "sar_gap_hi", "ceiling"]
for meth in ("tlp", "ce"):
    print(f"\n=== {meth.upper()} | P1 | alpha = 0.10 ===")
    print(view[(view.prompt == "P1") & (view.method == meth)][cols]
          .to_string(index=False, float_format=lambda v: f"{v:.3f}"))
inf = view[view.dem_infeasible]
print(f"\ndemographic group has SAR = 0 (no safe threshold) in {len(inf)} of {len(view)} slices")
print(f"ceiling (SAR = 1 for both groups: error rate already <= alpha, gap uninformative) "
      f"in {int(view.ceiling.sum())} of {len(view)} slices -- read those at alpha = 0.05")
print("\nOne shared threshold safe for BOTH groups (fallback when group labels are unavailable):")
print(view[view.prompt == "P1"][["dataset", "model", "method", "sar_ref", "sar_dem",
      "cons_cov_ref", "cons_cov_dem", "cons_cost_ref", "cons_cost_dem"]]
      .to_string(index=False, float_format=lambda v: f"{v:.3f}"))
print("cons_cost = automation each group gives up versus its own group-specific threshold")
print("\nOut-of-sample check: threshold set on 4/5 of the pairs, applied to the other 1/5:")
print(view[view.prompt == "P1"][["dataset", "model", "method", "sar_ref", "cf_cov_ref", "cf_err_ref",
      "sar_dem", "cf_cov_dem", "cf_err_dem"]].to_string(index=False, float_format=lambda v: f"{v:.3f}"))
print("cf_err well above 0.10 means the in-sample threshold would not stay safe on new cases")
print("\nTLP vs CE, same models and items -- does the confidence source change the conclusion?")
cmp = view[view.prompt == "P1"].pivot_table(index=["dataset", "model"], columns="method",
                                            values="sar_gap")
print(cmp.to_string(float_format=lambda v: f"{v:+.3f}"))

In [ ]:
c = con[con.alpha == 0.10].copy()
c["axis"] = c.dataset.str.contains("race").map({True: "race", False: "sex"})
print(c.groupby(["method", "axis", "condition"]).agg(
    slices=("d_gap", "size"), ceiling=("ceiling", "sum"),
    improves=("improves", "sum"), worsens=("worsens", "sum"),
    rules_out_meaningful=("rules_out_meaningful_improvement", "sum"),
    median_d_gap=("d_gap", "median"),
    costs_automation=("costs_automation", "sum"), adds_automation=("adds_automation", "sum"),
    median_d_sar_mean=("d_sar_mean", "median"), median_d_acc=("d_acc_mean", "median"))
    .to_string(float_format=lambda v: f"{v:+.3f}"))
print("\nd_gap < 0: the condition narrowed the gap (smallest improvement of interest "
      f"{analysis.SESOI:.2f}). d_sar_mean < 0: the condition lowered safe automation for the "
      "average case; costs_automation counts slices whose interval excludes 0. "
      "Ceiling slices cannot rule anything out.")
print("\nPer slice:")
print(c[["dataset", "model", "method", "condition", "d_gap", "d_gap_lo", "d_gap_hi",
         "d_sar_mean", "d_sar_mean_lo", "d_sar_mean_hi", "ceiling"]]
      .to_string(index=False, float_format=lambda v: f"{v:+.3f}"))
if len(res["letter_mass"]):
    print("\nTLP letter mass by model:")
    print(res["letter_mass"].groupby("model")[["median", "share_below_0.5"]].mean().round(3))

In [ ]:
from care import figures
for mdir in LOCAL.values():
    for meth in ("tlp", "ce"):             # one figure per confidence source; never overlaid
        cv = res["curves"]
        if len(cv[(cv.model == cfg.short(mdir)) & (cv.method == meth)]):
            print(figures.sar_curves(cv, cfg.short(mdir), meth))
print(figures.gap_forest(res["sar"]))
print(figures.tlp_vs_ce(res["sar"]))

## 8. Comparison with Boie et al. (J Med Syst 2026)

They report ECE of self-reported confidence on MedMCQA by **specialty**. The same
breakdown on our MedMCQA pairs, overall and per demographic group, places the
two stratifications side by side.

In [ ]:
es = res["ece_subject"]
if len(es):
    for meth, e in es[es.prompt == "P1"].groupby("method"):   # mean over models, per method
        t = e.groupby("subject")[["n", "accuracy", "ece"]].mean()
        print(f"\n=== ECE by subject, {meth.upper()}, P1 ===")
        print(t.sort_values("ece").to_string(float_format=lambda v: f"{v:.3f}"))

appx = analysis.run(datasets=list(APPX_DATA), models=list(APPENDIX),
               methods=("tlp", "ce"), verbose=False, tag="appendix") if APPENDIX else None
if appx is not None and len(appx["sar"]):
    a = appx["sar"][appx["sar"].alpha == 0.10]
    print("\nAppendix models, alpha = 0.10:")
    print(a[["dataset", "model", "method", "prompt", "sar_ref", "sar_dem", "sar_gap", "ceiling"]]
          .to_string(index=False, float_format=lambda v: f"{v:.3f}"))
    for al in (0.10, 0.05):
        ac = appx["contrast"][appx["contrast"].alpha == al]
        print(f"\nAppendix contrasts vs P1, alpha = {al}:")
        print(ac[["dataset", "model", "method", "condition", "d_gap", "d_gap_lo", "d_gap_hi", "d_sar_mean",
                  "d_sar_mean_lo", "d_sar_mean_hi", "ceiling"]]
              .to_string(index=False, float_format=lambda v: f"{v:+.3f}"))

### Every model on elicited confidence

CE is the one confidence source every model supports, so it is the common basis
for comparing all of them, local and API, on every pair file the API models ran.
TLP comparisons above are restricted to models that expose token probabilities.

In [ ]:
ce_all = analysis.run(datasets=list(APPX_DATA) or ["pairs_medqa_race"],
                 models=list(LOCAL.values()) + list(APPENDIX), methods=("ce",), verbose=False,
                 tag="ce_all_models")
if len(ce_all["sar"]):
    t = ce_all["sar"][(ce_all["sar"].alpha == 0.10) & (ce_all["sar"].prompt == "P1")]
    print(t[["dataset", "model", "acc_ref", "acc_dem", "sar_ref", "sar_dem", "sar_gap",
             "sar_gap_lo", "sar_gap_hi"]].sort_values("sar_gap", ascending=False)
          .to_string(index=False, float_format=lambda v: f"{v:.3f}"))

### Tables for the paper

Writes the LaTeX tables to `outputs/tables/` from the CSVs saved above.

In [ ]:
from care import tables
tables.main()